# Darcy 2D com obstaculo de baixa permeabilidade

Nesta aula, resolvemos um problema estacionario de escoamento em meio poroso
heterogeneo.

$$
\mathbf q = -\frac{k}{\mu}\nabla p,
\qquad
\nabla\cdot\mathbf q = 0.
$$

O dominio mede $50\,\mathrm{m}\times 50\,\mathrm{m}$. O obstaculo quadrado
central ocupa 15% da area do dominio. Na esquerda impomos a maior pressao, na
direita a menor pressao, e nas bordas superior e inferior impomos fluxo normal
nulo.

## Importando as dependencias

In [ ]:
from matplotlib.patches import Rectangle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pde
import scipy.sparse as sp
import scipy.sparse.linalg as spla

plt.rcParams.update(
    {
        "axes.grid": True,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "figure.figsize": (8, 4.5),
        "font.size": 11,
    }
)

## Classe do problema

In [ ]:
class DarcyObstacle2D:
    def __init__(
        self,
        length=50.0,
        shape=(80, 80),
        obstacle_area_fraction=0.15,
        permeability_matrix=1e-13,
        permeability_obstacle=1e-18,
        viscosity=1e-3,
        pressure_left=100e6,
        pressure_right=0.0,
    ):
        self.length = float(length)
        self.shape = tuple(shape)
        self.obstacle_area_fraction = float(obstacle_area_fraction)
        self.obstacle_side = self.length * np.sqrt(self.obstacle_area_fraction)
        self.permeability_matrix = float(permeability_matrix)
        self.permeability_obstacle = float(permeability_obstacle)
        self.viscosity = float(viscosity)
        self.pressure_left = float(pressure_left)
        self.pressure_right = float(pressure_right)
        self.grid = pde.CartesianGrid(
            [[0.0, self.length], [0.0, self.length]],
            self.shape,
            periodic=False,
        )

    def coordinates(self):
        positions_x, positions_y = self.grid.axes_coords
        return np.meshgrid(positions_x, positions_y, indexing="ij")

    def obstacle_bounds(self):
        start = 0.5 * (self.length - self.obstacle_side)
        end = 0.5 * (self.length + self.obstacle_side)
        return start, end

    def obstacle_mask(self):
        coordinates_x, coordinates_y = self.coordinates()
        start, end = self.obstacle_bounds()
        return (
            (coordinates_x >= start)
            & (coordinates_x <= end)
            & (coordinates_y >= start)
            & (coordinates_y <= end)
        )

    def permeability_data(self):
        permeability = self.permeability_matrix * np.ones(self.shape)
        permeability[self.obstacle_mask()] = self.permeability_obstacle
        return permeability

    def permeability_field(self):
        return pde.ScalarField(self.grid, data=self.permeability_data(), label="k")

    def mobility_data(self):
        return self.permeability_data() / self.viscosity

    @staticmethod
    def harmonic_mean(left_value, right_value):
        return 2.0 * left_value * right_value / (left_value + right_value)

    def cell_index(self, cell_x, cell_y):
        return cell_x * self.shape[1] + cell_y

    def assemble_pressure_system(self):
        num_cells_x, num_cells_y = self.shape
        step_x, step_y = map(float, self.grid.discretization)
        mobility = self.mobility_data()

        rows = []
        columns = []
        values = []
        right_hand_side = np.zeros(num_cells_x * num_cells_y)

        for cell_x in range(num_cells_x):
            for cell_y in range(num_cells_y):
                row = self.cell_index(cell_x, cell_y)
                diagonal = 0.0

                if cell_x == 0:
                    transmissibility = 2.0 * mobility[cell_x, cell_y] * step_y / step_x
                    diagonal += transmissibility
                    right_hand_side[row] += transmissibility * self.pressure_left
                else:
                    face_mobility = self.harmonic_mean(
                        mobility[cell_x - 1, cell_y],
                        mobility[cell_x, cell_y],
                    )
                    transmissibility = face_mobility * step_y / step_x
                    diagonal += transmissibility
                    rows.append(row)
                    columns.append(self.cell_index(cell_x - 1, cell_y))
                    values.append(-transmissibility)

                if cell_x == num_cells_x - 1:
                    transmissibility = 2.0 * mobility[cell_x, cell_y] * step_y / step_x
                    diagonal += transmissibility
                    right_hand_side[row] += transmissibility * self.pressure_right
                else:
                    face_mobility = self.harmonic_mean(
                        mobility[cell_x, cell_y],
                        mobility[cell_x + 1, cell_y],
                    )
                    transmissibility = face_mobility * step_y / step_x
                    diagonal += transmissibility
                    rows.append(row)
                    columns.append(self.cell_index(cell_x + 1, cell_y))
                    values.append(-transmissibility)

                if cell_y > 0:
                    face_mobility = self.harmonic_mean(
                        mobility[cell_x, cell_y - 1],
                        mobility[cell_x, cell_y],
                    )
                    transmissibility = face_mobility * step_x / step_y
                    diagonal += transmissibility
                    rows.append(row)
                    columns.append(self.cell_index(cell_x, cell_y - 1))
                    values.append(-transmissibility)

                if cell_y < num_cells_y - 1:
                    face_mobility = self.harmonic_mean(
                        mobility[cell_x, cell_y],
                        mobility[cell_x, cell_y + 1],
                    )
                    transmissibility = face_mobility * step_x / step_y
                    diagonal += transmissibility
                    rows.append(row)
                    columns.append(self.cell_index(cell_x, cell_y + 1))
                    values.append(-transmissibility)

                rows.append(row)
                columns.append(row)
                values.append(diagonal)

        matrix = sp.coo_matrix(
            (values, (rows, columns)),
            shape=(num_cells_x * num_cells_y, num_cells_x * num_cells_y),
        ).tocsr()
        return matrix, right_hand_side

    def solve_pressure(self):
        matrix, right_hand_side = self.assemble_pressure_system()
        pressure = spla.spsolve(matrix, right_hand_side).reshape(self.shape)
        return pde.ScalarField(self.grid, data=pressure, label="p [Pa]")

    def face_fluxes(self, pressure):
        pressure_data = np.asarray(pressure.data)
        num_cells_x, num_cells_y = self.shape
        step_x, step_y = map(float, self.grid.discretization)
        mobility = self.mobility_data()

        flux_x = np.zeros((num_cells_x + 1, num_cells_y))
        flux_y = np.zeros((num_cells_x, num_cells_y + 1))

        flux_x[0, :] = -mobility[0, :] * (
            pressure_data[0, :] - self.pressure_left
        ) / (0.5 * step_x)
        flux_x[-1, :] = -mobility[-1, :] * (
            self.pressure_right - pressure_data[-1, :]
        ) / (0.5 * step_x)

        for face_x in range(1, num_cells_x):
            face_mobility = self.harmonic_mean(
                mobility[face_x - 1, :],
                mobility[face_x, :],
            )
            flux_x[face_x, :] = -face_mobility * (
                pressure_data[face_x, :] - pressure_data[face_x - 1, :]
            ) / step_x

        for face_y in range(1, num_cells_y):
            face_mobility = self.harmonic_mean(
                mobility[:, face_y - 1],
                mobility[:, face_y],
            )
            flux_y[:, face_y] = -face_mobility * (
                pressure_data[:, face_y] - pressure_data[:, face_y - 1]
            ) / step_y

        return flux_x, flux_y

    def cell_center_velocity(self, pressure):
        flux_x, flux_y = self.face_fluxes(pressure)
        velocity_x = 0.5 * (flux_x[:-1, :] + flux_x[1:, :])
        velocity_y = 0.5 * (flux_y[:, :-1] + flux_y[:, 1:])
        speed = np.sqrt(velocity_x**2 + velocity_y**2)
        return velocity_x, velocity_y, speed

    def mass_balance(self, pressure):
        flux_x, _ = self.face_fluxes(pressure)
        _, step_y = map(float, self.grid.discretization)
        flow_left = np.sum(flux_x[0, :]) * step_y
        flow_right = np.sum(flux_x[-1, :]) * step_y
        imbalance = abs(flow_left - flow_right) / max(abs(flow_left), abs(flow_right))
        return flow_left, flow_right, imbalance


darcy = DarcyObstacle2D()
permeability = darcy.permeability_field()
pressure = darcy.solve_pressure()
velocity_x, velocity_y, speed = darcy.cell_center_velocity(pressure)
flow_left, flow_right, relative_imbalance = darcy.mass_balance(pressure)

## Campo de permeabilidade e solucao de pressao

In [ ]:
pd.DataFrame(
    {
        "quantidade": [
            "Lx = Ly",
            "Nx",
            "Ny",
            "Delta P",
            "k matriz",
            "k obstaculo",
            "lado do obstaculo",
            "fracao de area do obstaculo",
            "vazao pela esquerda",
            "vazao pela direita",
            "desbalanco relativo",
        ],
        "valor": [
            f"{darcy.length:g} m",
            darcy.shape[0],
            darcy.shape[1],
            f"{(darcy.pressure_left - darcy.pressure_right) / 1e6:g} MPa",
            f"{darcy.permeability_matrix:.1e} m^2",
            f"{darcy.permeability_obstacle:.1e} m^2",
            f"{darcy.obstacle_side:.2f} m",
            f"{100 * darcy.obstacle_area_fraction:.1f}%",
            f"{flow_left:.6e} m^2/s",
            f"{flow_right:.6e} m^2/s",
            f"{relative_imbalance:.2e}",
        ],
    }
)

In [ ]:
def plot_field_image(axis, grid, data, title, cmap="viridis", colorbar_label=None):
    image = axis.imshow(
        data.T,
        origin="lower",
        extent=[0.0, grid.axes_bounds[0][1], 0.0, grid.axes_bounds[1][1]],
        aspect="equal",
        cmap=cmap,
    )
    axis.set_xlabel("x [m]")
    axis.set_ylabel("y [m]")
    axis.set_title(title)
    plt.colorbar(image, ax=axis, label=colorbar_label)


def draw_obstacle(axis, model, color="white"):
    start, end = model.obstacle_bounds()
    axis.add_patch(
        Rectangle(
            (start, start),
            end - start,
            end - start,
            fill=False,
            edgecolor=color,
            linewidth=1.6,
        )
    )


fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

plot_field_image(
    axes[0],
    darcy.grid,
    np.log10(permeability.data),
    r"$\log_{10}(k)$",
    cmap="magma",
    colorbar_label=r"$\log_{10}(k/\mathrm{m^2})$",
)
plot_field_image(
    axes[1],
    darcy.grid,
    pressure.data / 1e6,
    "Pressao",
    cmap="viridis",
    colorbar_label="MPa",
)
plot_field_image(
    axes[2],
    darcy.grid,
    speed,
    "Modulo da velocidade de Darcy",
    cmap="cividis",
    colorbar_label="m/s",
)

for axis in axes:
    draw_obstacle(axis, darcy)

fig.tight_layout()
plt.show()

## Linhas de corrente e cortes

In [ ]:
positions_x, positions_y = darcy.grid.axes_coords
center_y_index = int(np.argmin(np.abs(positions_y - 0.5 * darcy.length)))
center_x_index = int(np.argmin(np.abs(positions_x - 0.5 * darcy.length)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

plot_field_image(
    axes[0],
    darcy.grid,
    pressure.data / 1e6,
    "Pressao e linhas de corrente",
    cmap="viridis",
    colorbar_label="MPa",
)
axes[0].streamplot(
    positions_x,
    positions_y,
    velocity_x.T,
    velocity_y.T,
    color="white",
    density=1.2,
    linewidth=0.8,
)
draw_obstacle(axes[0], darcy)

axes[1].plot(
    positions_x,
    pressure.data[:, center_y_index] / 1e6,
    label=rf"$p(x,{positions_y[center_y_index]:.1f})$",
)
axes[1].plot(
    positions_y,
    pressure.data[center_x_index, :] / 1e6,
    label=rf"$p({positions_x[center_x_index]:.1f},y)$",
)
axes[1].set_xlabel("coordenada [m]")
axes[1].set_ylabel("pressao [MPa]")
axes[1].set_title("Cortes pelo centro do dominio")
axes[1].legend()

fig.tight_layout()
plt.show()